# Online Retail: Data Cleaning, EDA and Slow-Moving Inventory Analysis

**Stack:** Python, Pandas, NumPy, Matplotlib, Seaborn, Jupyter
**Data:** UCI Online Retail (about 540K transactions, Dec 2010 to Dec 2011, UK-based online gift retailer)

**Flow:** load -> clean (with an audit log) -> EDA -> slow-moving inventory classification -> findings.

**Before running:** download the dataset and place it in `data/` as `Online Retail.xlsx` (UCI) or `OnlineRetail.csv` (Kaggle copy).
Every number in the final summary is computed by the notebook, so quote *those* numbers in your resume and README.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_columns = 50

DATA_DIR = Path("data")
OUT_DIR = Path("outputs")
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ---- analysis parameters (tune here) ----
DORMANT_DAYS = 180      # no sale in this many days (before the dataset ends) => dormant
OUTLIER_QTY = 50_000    # single-line quantity treated as an order-and-cancel outlier

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 1. Load data

In [ ]:
xlsx = DATA_DIR / "Online Retail.xlsx"
csv = DATA_DIR / "OnlineRetail.csv"
dtypes = {"InvoiceNo": str, "StockCode": str}

if xlsx.exists():
    raw = pd.read_excel(xlsx, dtype=dtypes)          # needs openpyxl
elif csv.exists():
    raw = pd.read_csv(csv, dtype=dtypes, encoding="ISO-8859-1")
else:
    raise FileNotFoundError("Put 'Online Retail.xlsx' or 'OnlineRetail.csv' inside the data/ folder.")

print(f"Rows: {len(raw):,}   Columns: {raw.shape[1]}")
raw.head()

In [ ]:
raw.info()
print("\nMissing values per column:")
print(raw.isna().sum())
raw.describe(include="all").T

## 2. Cleaning
Each step is logged so the cleaning is auditable. Order: de-duplicate -> separate cancellations -> drop non-product codes -> drop invalid rows -> drop order-and-cancel outliers.

*Missing `CustomerID` rows are kept on purpose: this analysis is product-level, not customer-level.*

In [ ]:
log = {"raw_rows": len(raw)}

# 2.1 exact duplicate rows
df = raw.drop_duplicates().copy()
log["exact_duplicates_removed"] = log["raw_rows"] - len(df)

# 2.2 tidy types / text
df["InvoiceNo"] = df["InvoiceNo"].astype(str).str.strip()
df["StockCode"] = df["StockCode"].astype(str).str.strip().str.upper()
df["Description"] = df["Description"].str.strip()
df["Country"] = df["Country"].str.strip()
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print(f"Duplicates removed: {log['exact_duplicates_removed']:,}")

In [ ]:
# 2.3 separate cancellations (invoice numbers starting with 'C')
is_cancel = df["InvoiceNo"].str.startswith("C")
cancellations = df[is_cancel].copy()
sales = df[~is_cancel].copy()
log["cancellation_rows_separated"] = len(cancellations)
print(f"Cancellation rows: {len(cancellations):,}   Sales rows: {len(sales):,}")

In [ ]:
# 2.4 non-product stock codes (postage, fees, manual adjustments, samples, gift vouchers...)
NON_PRODUCT = {"POST", "D", "M", "DOT", "CRUK", "C2", "BANK CHARGES", "S", "B",
               "AMAZONFEE", "PADS", "ADJUST", "ADJUST2", "TEST001", "TEST002"}

def is_non_product(codes):
    return codes.isin(NON_PRODUCT) | codes.str.startswith("GIFT_")

np_mask = is_non_product(sales["StockCode"])
print("Non-product rows in sales, by code:")
print(sales.loc[np_mask, "StockCode"].value_counts().head(15))

log["non_product_rows_removed"] = int(np_mask.sum())
sales = sales[~np_mask].copy()
cancellations = cancellations[~is_non_product(cancellations["StockCode"])].copy()

In [ ]:
# 2.5 invalid sales rows (zero/negative price or quantity: write-offs, adjustments)
bad = (sales["Quantity"] <= 0) | (sales["UnitPrice"] <= 0)
log["invalid_qty_or_price_removed"] = int(bad.sum())
sales = sales[~bad].copy()
print(f"Invalid rows removed: {int(bad.sum()):,}")

In [ ]:
# 2.6 order-and-cancel outliers: a huge order that is fully reversed distorts top-product revenue
big_sales = sales[sales["Quantity"] >= OUTLIER_QTY]
big_cancels = cancellations[cancellations["Quantity"].abs() >= OUTLIER_QTY]
print("Outlier sales lines:")
display(big_sales[["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "InvoiceDate"]])
print("Matching cancellation lines:")
display(big_cancels[["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "InvoiceDate"]])

log["outlier_sales_rows_removed"] = len(big_sales)
log["outlier_cancel_rows_removed"] = len(big_cancels)
sales = sales.drop(big_sales.index)
cancellations = cancellations.drop(big_cancels.index)

In [ ]:
# 2.7 feature engineering
sales["Revenue"] = sales["Quantity"] * sales["UnitPrice"]
sales["Month"] = sales["InvoiceDate"].dt.to_period("M")
sales["Weekday"] = sales["InvoiceDate"].dt.day_name()
sales["Hour"] = sales["InvoiceDate"].dt.hour
cancellations["Value"] = cancellations["Quantity"] * cancellations["UnitPrice"]

log["clean_sales_rows"] = len(sales)
log_df = pd.Series(log, name="rows").to_frame()
display(log_df)

print(f"\nClean sales rows : {len(sales):,}")
print(f"Unique products  : {sales['StockCode'].nunique():,}")
print(f"Unique invoices  : {sales['InvoiceNo'].nunique():,}")
print(f"Date range       : {sales['InvoiceDate'].min():%Y-%m-%d} -> {sales['InvoiceDate'].max():%Y-%m-%d}")
print(f"Total revenue    : {sales['Revenue'].sum():,.0f}")

## 3. Cancellations

In [ ]:
cancel_value = cancellations["Value"].abs().sum()
print(f"Cancellation lines : {len(cancellations):,}")
print(f"Cancelled value    : {cancel_value:,.0f}  ({cancel_value / sales['Revenue'].sum():.1%} of sales revenue)")

top_cancel = (cancellations.groupby(["StockCode", "Description"])["Quantity"].sum().abs()
              .sort_values(ascending=False).head(10))
top_cancel

## 4. Exploratory data analysis

### 4.1 Monthly revenue and seasonality

In [ ]:
monthly = sales.groupby("Month")["Revenue"].sum()
total_rev = monthly.sum()

sepnov_rev = sales.loc[sales["InvoiceDate"].dt.month.isin([9, 10, 11]), "Revenue"].sum()
sepnov_share = sepnov_rev / total_rev
y2011 = sales[(sales["InvoiceDate"] >= "2011-01-01") & (sales["InvoiceDate"] < "2011-12-01")]
sepnov_share_2011 = sepnov_rev / y2011["Revenue"].sum()

print(f"Sep-Nov share of ALL revenue in dataset      : {sepnov_share:.1%}")
print(f"Sep-Nov share of Jan-Nov 2011 revenue        : {sepnov_share_2011:.1%}")

colors = ["#d95f02" if p.month in (9, 10, 11) else "#7f9fbf" for p in monthly.index]
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(monthly.index.astype(str), monthly.values, color=colors)
ax.set_title("Monthly revenue (Sep-Nov highlighted; Dec 2011 is a partial month)")
ax.set_ylabel("Revenue")
plt.xticks(rotation=45)
savefig("01_monthly_revenue")

### 4.2 Country mix

In [ ]:
country_rev = sales.groupby("Country")["Revenue"].sum().sort_values(ascending=False)
uk_share = country_rev["United Kingdom"] / country_rev.sum()
print(f"United Kingdom share of revenue: {uk_share:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
country_rev.head(10).sort_values().plot.barh(ax=axes[0], color="#4c78a8")
axes[0].set_title("Top 10 countries by revenue")
country_rev.drop("United Kingdom").head(10).sort_values().plot.barh(ax=axes[1], color="#59a14f")
axes[1].set_title("Top 10 excluding the UK")
for a in axes:
    a.set_xlabel("Revenue")
savefig("02_country_mix")

### 4.3 Top products by revenue (after outlier removal)

In [ ]:
desc = (sales.groupby("StockCode")["Description"]
        .agg(lambda s: s.dropna().mode().iloc[0] if s.notna().any() else ""))

top_products = (sales.groupby("StockCode")["Revenue"].sum()
                .sort_values(ascending=False).head(10))
labels = [f"{c} | {desc[c][:28]}" for c in top_products.index]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(labels[::-1], top_products.values[::-1], color="#e15759")
ax.set_title("Top 10 products by revenue")
ax.set_xlabel("Revenue")
savefig("03_top_products")

### 4.4 Weekday x hour demand

In [ ]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
heat = (sales.groupby(["Weekday", "Hour"])["InvoiceNo"].nunique()
        .unstack(fill_value=0))
heat = heat.reindex([d for d in order if d in heat.index])

fig, ax = plt.subplots(figsize=(12, 4.5))
sns.heatmap(heat, cmap="YlOrRd", linewidths=0.3, cbar_kws={"label": "Unique invoices"}, ax=ax)
ax.set_title("Demand by weekday and hour (unique invoices)")
savefig("04_weekday_hour_heatmap")

print("Weekdays with no sales:", [d for d in order if d not in heat.index])
print("Busiest weekday:", heat.sum(axis=1).idxmax(), "| Busiest hour:", heat.sum(axis=0).idxmax())

## 5. Slow-moving inventory analysis
Each product (StockCode) is classified by **days since its last sale**, measured from the last date in the data.

| Status | Days since last sale |
|---|---|
| Active | 0-30 |
| Slow | 31-90 |
| Very slow | 91 to `DORMANT_DAYS` |
| Dormant | more than `DORMANT_DAYS` |

*Caveat: the data covers about 13 months and has no stock levels. "Dormant" means "no sales in the window", so seasonal items and discontinued lines need a human check before any clearance decision.*

In [ ]:
snapshot = sales["InvoiceDate"].max().normalize()

prod = (sales.groupby("StockCode")
        .agg(first_sale=("InvoiceDate", "min"),
             last_sale=("InvoiceDate", "max"),
             orders=("InvoiceNo", "nunique"),
             units=("Quantity", "sum"),
             revenue=("Revenue", "sum"))
        .join(desc.rename("Description")))

prod["days_since_last_sale"] = (snapshot - prod["last_sale"].dt.normalize()).dt.days

labels_status = ["Active", "Slow", "Very slow", "Dormant"]
prod["status"] = pd.cut(prod["days_since_last_sale"],
                        bins=[-1, 30, 90, DORMANT_DAYS, np.inf],
                        labels=labels_status)

summary = (prod.groupby("status", observed=True)
           .agg(products=("revenue", "size"), revenue=("revenue", "sum")))
summary["pct_products"] = summary["products"] / summary["products"].sum() * 100
summary["pct_revenue"] = summary["revenue"] / summary["revenue"].sum() * 100
summary = summary.reindex(labels_status)
display(summary)

n_products = len(prod)
dormant = prod[prod["status"] == "Dormant"]
dormant_pct_products = len(dormant) / n_products
dormant_pct_revenue = dormant["revenue"].sum() / prod["revenue"].sum()
print(f"Products classified : {n_products:,}")
print(f"Dormant products    : {len(dormant):,} ({dormant_pct_products:.1%} of products, "
      f"{dormant_pct_revenue:.1%} of revenue)")

In [ ]:
# Sensitivity: how the dormant count changes with the threshold
rows = []
for d in [60, 90, 120, 180, 270]:
    m = prod["days_since_last_sale"] > d
    rows.append({"threshold_days": d, "dormant_products": int(m.sum()),
                 "pct_products": m.mean() * 100,
                 "pct_revenue": prod.loc[m, "revenue"].sum() / prod["revenue"].sum() * 100})
pd.DataFrame(rows)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

x = np.arange(len(labels_status))
w = 0.38
axes[0].bar(x - w / 2, summary["pct_products"], w, label="% of products", color="#4c78a8")
axes[0].bar(x + w / 2, summary["pct_revenue"], w, label="% of revenue", color="#f58518")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels_status)
axes[0].set_ylabel("Percent")
axes[0].set_title("Product share vs revenue share by status")
axes[0].legend()

axes[1].hist(prod["days_since_last_sale"], bins=40, color="#7f9fbf")
axes[1].axvline(DORMANT_DAYS, color="red", ls="--", label=f"Dormant > {DORMANT_DAYS} days")
axes[1].set_title("Days since last sale (per product)")
axes[1].set_xlabel("Days")
axes[1].legend()
savefig("05_slow_moving_status")

In [ ]:
# Clearance review list: dormant products, longest-idle first
clearance = (dormant.assign(avg_unit_price=lambda d: d["revenue"] / d["units"])
             .sort_values("days_since_last_sale", ascending=False)
             .reset_index()
             [["StockCode", "Description", "first_sale", "last_sale", "days_since_last_sale",
               "orders", "units", "revenue", "avg_unit_price"]])
clearance.to_csv(OUT_DIR / "clearance_candidates.csv", index=False)
prod.reset_index().to_csv(OUT_DIR / "product_status_all.csv", index=False)
print(f"Saved {len(clearance):,} clearance candidates -> {OUT_DIR / 'clearance_candidates.csv'}")
clearance.head(15)

## 6. Key findings (auto-generated from the results above)

In [ ]:
top_prod_code = top_products.index[0]
print(f"""
DATA CLEANING
- Started with {log['raw_rows']:,} transactions; removed {log['exact_duplicates_removed']:,} exact duplicates.
- Separated {log['cancellation_rows_separated']:,} cancellation lines; removed {log['non_product_rows_removed']:,} non-product rows
  and {log['invalid_qty_or_price_removed']:,} invalid (zero/negative) rows.
- Excluded {log['outlier_sales_rows_removed']} order-and-cancel outlier sale(s) and {log['outlier_cancel_rows_removed']} matching cancellation(s).
- Clean sales table: {log['clean_sales_rows']:,} rows.

SLOW-MOVING INVENTORY
- Classified {n_products:,} products by days since last sale.
- {len(dormant):,} dormant products (> {DORMANT_DAYS} days): {dormant_pct_products:.0%} of products, {dormant_pct_revenue:.0%} of revenue.

DEMAND PATTERNS
- Sep-Nov = {sepnov_share:.0%} of all revenue ({sepnov_share_2011:.0%} of Jan-Nov 2011).
- UK = {uk_share:.0%} of revenue.
- Top product by revenue: {top_prod_code} ({desc[top_prod_code]}).
""")